# 🤖 Google Colab SSH & Multi-Agent Worker Node

Dieses Notebook richtet vollen **SSH-Zugriff** (via `colab-ssh` & Tailscale SSH) sowie den **Multi-Agenten CUA AI Desktop** ein.

## 🔑 1. SSH-Zugriff einrichten (via colab-ssh GitHub Repo)
Verwendet das offizielle [`colab-ssh`](https://github.com/WassimBenzarti/colab-ssh) Repository mit Cloudflare Tunnel für weltweiten SSH-Zugriff ohne Portweiterleitungen.

In [ ]:
!pip install -q colab_ssh --upgrade
from colab_ssh import launch_ssh_cloudflared

# Setze dein Wunschpasswort für die SSH-Sitzung:
SSH_PASSWORD = "ColabSecret123!"  # <- Hier dein Passwort eintragen

launch_ssh_cloudflared(password=SSH_PASSWORD)

## 🔗 2. Alternative: Direktes Tailscale SSH (Im privaten Mesh-Netzwerk)
Ermöglicht passwortloses SSH direkt über `ssh root@colab-worker-1` von deinem Mac oder Debian VPS.

In [ ]:
TAILSCALE_AUTH_KEY = ""  # Optional: tskey-auth-...
NODE_NAME = "colab-worker-1"

!curl -fsSL https://tailscale.com/install.sh | sh
!apt-get update -qq && apt-get install -y -qq openssh-server
!service ssh start

import subprocess, time
subprocess.Popen(["tailscaled", "--tun=userspace-networking", "--socks5-server=localhost:1055", "--outbound-http-proxy-listen=localhost:1055"])
time.sleep(3)

if TAILSCALE_AUTH_KEY:
    !tailscale up --authkey=$TAILSCALE_AUTH_KEY --hostname=$NODE_NAME --ssh --accept-routes
else:
    !tailscale up --hostname=$NODE_NAME --ssh --accept-routes

## 🖥️ 3. Multi-Worker Virtual Displays & CUA Desktop starten

In [ ]:
!apt-get install -y -qq xvfb fluxbox x11vnc novnc websockify chromium-browser python3-pip
!pip install -q fastapi uvicorn requests pyautogui mss Pillow pydantic

import os, subprocess, time
workers = [
    {"id": 1, "display": ":1", "cdp_port": 9222, "vnc_port": 5901, "novnc_port": 6081},
    {"id": 2, "display": ":2", "cdp_port": 9223, "vnc_port": 5902, "novnc_port": 6082},
    {"id": 3, "display": ":3", "cdp_port": 9224, "vnc_port": 5903, "novnc_port": 6083}
]

for w in workers:
    d_dir = f"/tmp/bot_profile_{w['id']}"
    os.makedirs(d_dir, exist_ok=True)
    subprocess.Popen(["Xvfb", w["display"], "-screen", "0", "1280x800x24"])
    time.sleep(1)
    subprocess.Popen(["fluxbox"], env={**os.environ, "DISPLAY": w["display"]})
    subprocess.Popen(["x11vnc", "-display", w["display"], "-rfbport", str(w["vnc_port"]), "-shared", "-forever", "-nopw", "-bg"])
    subprocess.Popen(["websockify", "--web=/usr/share/novnc/", str(w["novnc_port"]), f"localhost:{w['vnc_port']}"])
    subprocess.Popen([
        "chromium-browser", f"--user-data-dir={d_dir}", "--no-sandbox", "--disable-dev-shm-usage",
        "--disable-gpu", f"--remote-debugging-port={w['cdp_port']}", "--remote-debugging-address=0.0.0.0",
        "--window-size=1280,800", "about:blank"
    ], env={**os.environ, "DISPLAY": w["display"]})
    print(f"✅ Bot Worker {w['id']} gestartet: CDP Port {w['cdp_port']} | Live noVNC Port {w['novnc_port']}")

print("\n🎉 Alle Bot-Worker sind aktiv!")

## 🩺 4. Health & Control API (Port 9090)

In [ ]:
from fastapi import FastAPI
import uvicorn, psutil, threading

app = FastAPI(title="Colab Bot Worker Node")

@app.get("/health")
def health():
    return {
        "status": "healthy",
        "node": NODE_NAME,
        "workers": workers,
        "ram_used_percent": psutil.virtual_memory().percent,
        "cpu_percent": psutil.cpu_percent()
    }

threading.Thread(target=lambda: uvicorn.run(app, host="0.0.0.0", port=9090, log_level="warning"), daemon=True).start()
print("🩺 Health API aktiv auf Port 9090!")

## 🟢 5. Keep-Alive Schleife

In [ ]:
import datetime, time
print("🟢 Worker-Knoten läuft aktiv...")
try:
    while True:
        time.sleep(60)
        now = datetime.datetime.now().strftime("%H:%M:%S")
        print(f"[{now}] Heartbeat: Aktiv | RAM: {psutil.virtual_memory().percent}%")
except KeyboardInterrupt:
    print("Beendet.")